In [5]:
#Import libraries
import requests
import pandas as pd
from pathlib import Path
import json
from datetime import timedelta

In [6]:
#Folder Structure
cwd = Path.cwd()

PROJECT_ROOT = cwd.parent if cwd.name == "Notebooks" else cwd

RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Current folder:", cwd)
print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)

Current folder: C:\Users\iniob\Toni_projects\Hydrogen Project\Notebooks
Project root: C:\Users\iniob\Toni_projects\Hydrogen Project
Raw data folder: C:\Users\iniob\Toni_projects\Hydrogen Project\Data\Raw
Processed data folder: C:\Users\iniob\Toni_projects\Hydrogen Project\Data\Processed


In [7]:
BASE_URL = 'https://api.carbonintensity.org.uk'

In [8]:
#Function to pull one day
def fetch_carbon_intensity_for_date(date_str):
    '''
    Fetch UK carbon intensity data for one date.

    date_str in YYYY-MM-DD format.
    '''
    url = f'{BASE_URL}/intensity/date/{date_str}'

    response = requests.get(url)
    response.raise_for_status()

    raw_data=response.json()

    #Save raw JSONN
    raw_path =RAW_DIR/f'carbon_intensity_{date_str}.json'
    with open(raw_path, 'w') as f:
        json.dump(raw_data, f, indent=2)

    records = []

    #Store data
    for item in raw_data["data"]:
        records.append({
            "date_requested": date_str,
            "from_utc": item["from"],
            "to_utc": item["to"],
            "forecast_gco2_per_kwh": item["intensity"].get("forecast"),
            "actual_gco2_per_kwh": item["intensity"].get("actual"),
            "index": item["intensity"].get("index")
        })

    df = pd.DataFrame(records)

    #Fix to datetime and add in timings
    df["from_utc"] = pd.to_datetime(df["from_utc"], utc=True)
    df["to_utc"] = pd.to_datetime(df["to_utc"], utc=True)
    
    df["from_uk"] = df["from_utc"].dt.tz_convert("Europe/London")
    df["to_uk"] = df["to_utc"].dt.tz_convert("Europe/London")
    
    df["date_uk"] = df["from_uk"].dt.date
    df["hour_uk"] = df["from_uk"].dt.hour
    df["time_uk"] = df["from_uk"].dt.time
    
    return df

In [9]:
#Pull the last 7 complete days, did this on 29.06.2026
end_date = pd.Timestamp.now(tz="Europe/London").date() - timedelta(days=1)
start_date = end_date - timedelta(days=6)

all_days= []

current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime('%Y-%m-%d')
    print('Fetching: ', date_str)

    df_day = fetch_carbon_intensity_for_date(date_str)
    all_days.append(df_day)

    current_date += timedelta(days=1)

df_week = pd.concat(all_days, ignore_index=True)


Fetching:  2026-06-22
Fetching:  2026-06-23
Fetching:  2026-06-24
Fetching:  2026-06-25
Fetching:  2026-06-26
Fetching:  2026-06-27
Fetching:  2026-06-28


In [10]:
#Save the processed data

processed_path = PROCESSED_DIR / "carbon_intensity_last_7_complete_days.csv"
df_week.to_csv(processed_path, index=False)

print("Saved to:", processed_path)
print("Rows:", len(df_week))

df_week.head()

Saved to: C:\Users\iniob\Toni_projects\Hydrogen Project\Data\Processed\carbon_intensity_last_7_complete_days.csv
Rows: 336


,date_requested,from_utc,to_utc,forecast_gco2_per_kwh,actual_gco2_per_kwh,index,from_uk,to_uk,date_uk,hour_uk,time_uk
0,2026-06-22,2026-06-21 23:00:00+00:00,2026-06-21 23:30:00+00:00,193,189,high,2026-06-22 00:00:00+01:00,2026-06-22 00:30:00+01:00,2026-06-22,0,00:00:00
1,2026-06-22,2026-06-21 23:30:00+00:00,2026-06-22 00:00:00+00:00,188,187,high,2026-06-22 00:30:00+01:00,2026-06-22 01:00:00+01:00,2026-06-22,0,00:30:00
2,2026-06-22,2026-06-22 00:00:00+00:00,2026-06-22 00:30:00+00:00,196,182,high,2026-06-22 01:00:00+01:00,2026-06-22 01:30:00+01:00,2026-06-22,1,01:00:00
3,2026-06-22,2026-06-22 00:30:00+00:00,2026-06-22 01:00:00+00:00,190,179,high,2026-06-22 01:30:00+01:00,2026-06-22 02:00:00+01:00,2026-06-22,1,01:30:00
4,2026-06-22,2026-06-22 01:00:00+00:00,2026-06-22 01:30:00+00:00,189,179,high,2026-06-22 02:00:00+01:00,2026-06-22 02:30:00+01:00,2026-06-22,2,02:00:00


In [11]:
df_week.shape #Expecting 336 rows

(336, 11)

In [12]:
df_week.isna().sum()

date_requested           0
from_utc                 0
to_utc                   0
forecast_gco2_per_kwh    0
actual_gco2_per_kwh      0
index                    0
from_uk                  0
to_uk                    0
date_uk                  0
hour_uk                  0
time_uk                  0
dtype: int64